#Connect to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


#Unzip data

In [ ]:
!cp gdrive/My\ Drive/MASTER_2024_2025/DOG_WOLF.zip /content/
!unzip DOG_WOLF.zip -d DATI/

#Import

In [ ]:
###########################
# IMPORT
import scipy.io as sio
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
import shutil
import copy
import random
from random import shuffle
import matplotlib.pyplot as plt
######################################

#Check device


In [ ]:
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

2.6.0+cu124
True
Tesla T4
cuda:0


# Create train - test- val

In [ ]:
def split_train_val_test(list_files, percentage_train):
  shuffle(list_files)
  n_train = int(len(list_files)*percentage_train)
  n_remain = len(list_files) - n_train
  val_test = int(n_remain/2)

  train_files = list_files[:n_train]
  remain = list_files[n_train:]

  val_files = remain[:val_test]
  test_files = remain[val_test:]

  return train_files, val_files, test_files



basePath = '/content/DATI/DOG_WOLF/data/'

list_dogs = os.listdir(basePath + 'dogs/')
list_wolves = os.listdir(basePath + 'wolves/')

percentage_train = 0.8
train_files_dogs, val_files_dogs, test_files_dogs = split_train_val_test(list_dogs, percentage_train)
train_files_wolves, val_files_wolves, test_files_wolves = split_train_val_test(list_wolves, percentage_train)

outPath = '/content/DATI/SplitDog_wolf/data/'
path_train_dogs =  outPath + 'train/dogs/'
path_val_dogs =  outPath + 'val/dogs/'

path_train_wolves =  outPath + 'train/wolves/'
path_val_wolves =  outPath + 'val/wolves/'

path_test =  outPath + 'test/'

try:
  os.makedirs(path_train_dogs)
  os.makedirs(path_val_dogs)
  os.makedirs(path_train_wolves)
  os.makedirs(path_val_wolves)
  os.makedirs(path_test)
except:
  pass

for file in train_files_dogs:
  shutil.copyfile(basePath + 'dogs/' + file, path_train_dogs + file)

for file in train_files_wolves:
  shutil.copyfile(basePath + 'wolves/' + file, path_train_wolves + file)

for file in val_files_dogs:
  shutil.copyfile(basePath + 'dogs/' + file, path_val_dogs + file)

for file in val_files_wolves:
  shutil.copyfile(basePath + 'wolves/' + file, path_val_wolves + file)


list_classes = []
for file in test_files_dogs:
  shutil.copyfile(basePath + 'dogs/' + file, path_test + file)
  list_classes.append('dogs')

for file in test_files_wolves:
  shutil.copyfile(basePath + 'wolves/' + file, path_test + file)
  list_classes.append('wolves')

test_file = pd.DataFrame()
test_file['files'] = test_files_dogs + test_files_wolves
test_file['label'] = list_classes

test_file.to_csv('testLabels.csv', index = False)

#Load Data

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}


data_dir = '/content/DATI/SplitDog_wolf/data/'
#data_transforms['train']
appoggio = datasets.ImageFolder(os.path.join(data_dir, 'train'), data_transforms['train'])
appoggio.samples[100]
appoggio.classes

image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x),data_transforms[x]) for x in ['train', 'val']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=batch_size, shuffle=True, num_workers=4) for x in ['train', 'val']}


dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
class_names = image_datasets['train'].classes

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
class_names = (image_datasets['train'].classes)
print(class_names)
print(image_datasets['val'].classes)

In [ ]:
img, cl = appoggio.__getitem__(100)

print(img.shape)

plt.imshow(img.permute([1,2,0]))
plt.show()

''''
for x,y in dataloaders['train']:
  print(x.shape)
  print(y.shape)
'''

# Network for features extraction

In [ ]:
model_ft = torchvision.models.alexnet(pretrained=True)
#remove classifier
model_ft.classifier = nn.Sequential()
#img = torch.rand(1,3,224,227)
#outputs = model_ft(img)
#print(outputs.shape)
model_ft.eval()

'''
#remove the last layer
model_ft = torchvision.models.alexnet(pretrained=True)
new_classifier = nn.Sequential(*list(model_ft.classifier.children())[:-1])
model_ft.classifier = new_classifier

img = torch.rand(1,3,224,227)
outputs = model_ft(img)
print(outputs.shape)
'''
outfeatures = []
list_label = []
#extract features for each element in the training set
for i in range(0, len(image_datasets['train'].samples)):
  #read image
  img, label = image_datasets['train'].__getitem__(i)
  #img -> 3,224,224
  features = model_ft(img.unsqueeze(0))  #9216

  outfeatures.append((np.squeeze(features.detach()).numpy()))
  list_label.append(label)

In [ ]:
TRAIN_features = pd.DataFrame(outfeatures)
colonne =[('f' + str(i)) for i in list(range(0, 9216 )) ]
TRAIN_features.columns =  colonne
TRAIN_features['label'] = list_label
print(TRAIN_features.shape) #csv file

In [ ]:
outfeatures = []
list_label = []
#extract features for each element in the training set
for i in range(0, len(image_datasets['val'].samples)):
  #read image
  img, label = image_datasets['val'].__getitem__(i)
  #img -> 3,224,224
  features = model_ft(img.unsqueeze(0))  #9216

  outfeatures.append((np.squeeze(features.detach()).numpy()))
  list_label.append(label)